# Inspecting `data/sessions.db`

This SQLite file backs two things in the LangGraph pipeline:

1. **Conversation history** — `message_store` (written by `SQLChatMessageHistory`, read via `session_store.get_history`)
2. **Graph checkpoints** — `checkpoints` / `writes` (written by LangGraph's `SqliteSaver`; this is where `AgentState` — workflow progress, pending context-switches, etc. — is persisted per `thread_id` = `session_id`)

There's also a `workflow_state` table — that's a **leftover from the pre-LangGraph implementation** and is no longer written to; the checkpointer's `checkpoints` table replaced it.

In [16]:
import sqlite3
import json
import pandas as pd

DB_PATH = "../data/sessions.db"
conn = sqlite3.connect(DB_PATH)

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
tables

,name
0,workflow_state
1,message_store
2,checkpoints
3,writes


## Listing all users / sessions

`session_id` is the same as the LangGraph `thread_id`, and in the CLI it's set to the `user_id` you enter at the Chat Mode prompt — so listing distinct `session_id` values shows you every user who has chatted.

In [17]:
session_ids = pd.read_sql(
    "SELECT session_id, COUNT(*) AS message_count FROM message_store GROUP BY session_id ORDER BY message_count DESC",
    conn,
)
session_ids

,session_id,message_count
0,cli_session,38
1,test_wf_f48938,8
2,test_wf_995509,8
3,test_lg_91701012,4
4,ankit,4
5,alice,4
6,test_guard_a47ad6,2
7,test_cache_37a296,2
8,test_cache2_75ed68,2
9,shaanu,2


## 1. Conversation history (`message_store`)

One row per message (human or AI), keyed by `session_id`. The `message` column is a JSON-serialized LangChain message.

In [ ]:
messages_df = pd.read_sql("SELECT * FROM message_store", conn)
print(f"{len(messages_df)} messages across {messages_df['session_id'].nunique()} session(s)")
messages_df.head(20)

In [ ]:
# Pretty-print one session's conversation
SESSION_ID = messages_df["session_id"].iloc[0] if not messages_df.empty else None
print(f"Session: {SESSION_ID}\n")

for _, row in messages_df[messages_df["session_id"] == SESSION_ID].iterrows():
    msg = json.loads(row["message"])
    role = msg.get("type", "?")
    content = msg.get("data", {}).get("content", "")
    print(f"[{role:9}] {content}\n")

## 2. Graph checkpoints (`checkpoints` / `writes`)

Written by LangGraph's `SqliteSaver`. `thread_id` == `session_id`. Each row is a snapshot of `AgentState` at a point in the graph's execution — this is how workflow progress (e.g. mid-`book_desk`, collected params, pending context-switches) survives across separate `query_rag_simple()` calls and process restarts.

The `checkpoint` column is msgpack-serialized (LangGraph's `JsonPlusSerializer`), so we decode it with the same serializer the checkpointer uses.

In [ ]:
checkpoints_df = pd.read_sql(
    "SELECT thread_id, checkpoint_ns, checkpoint_id, parent_checkpoint_id, type FROM checkpoints",
    conn,
)
print(f"{len(checkpoints_df)} checkpoints across {checkpoints_df['thread_id'].nunique()} thread(s)")
checkpoints_df.head(20)

In [ ]:
# Decode the latest checkpoint for a given thread_id (= session_id) using LangGraph's own serde,
# and inspect the persisted AgentState fields directly.
from langgraph.checkpoint.sqlite import SqliteSaver

THREAD_ID = SESSION_ID  # change to inspect a different user/session

raw_conn = sqlite3.connect(DB_PATH, check_same_thread=False)
saver = SqliteSaver(raw_conn)

config = {"configurable": {"thread_id": THREAD_ID}}
snapshot = saver.get_tuple(config)

if snapshot is None:
    print(f"No checkpoint found for thread_id={THREAD_ID!r}")
else:
    state = snapshot.checkpoint["channel_values"]
    # vector_store / large fields are intentionally NOT in AgentState (kept out of persistence) —
    # what's left is exactly the cross-turn fields: workflow progress, routing, RAG scratch state, etc.
    for key in sorted(state.keys()):
        value = state[key]
        # Trim long fields (history, contexts) for readability
        if isinstance(value, list) and len(value) > 3:
            value = f"[{len(value)} items] {value[:1]} ..."
        print(f"{key:24} = {value}")

## 3. (Legacy, unused) `workflow_state` table

Pre-LangGraph leftover — the old hand-rolled workflow persistence. Nothing writes to it anymore (workflow progress now lives in the checkpointer's `AgentState` snapshots above). Shown here only for completeness / migration history.

In [ ]:
pd.read_sql("SELECT * FROM workflow_state", conn)

In [ ]:
conn.close()
raw_conn.close()